## Colorado Spills Project
### incorporating rurality data
### Date: 2025-08-06


In [1]:
import pandas as pd
import geopandas as gpd
import statsmodels.formula.api as smf
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()

# Connect to your database
engine = create_engine(f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/colorado_spills")

# Load data with geometry preserved
spills = gpd.read_postgis("SELECT * FROM spills_with_ruca", engine, geom_col='geometry')


In [2]:
def classify_rurality(ruca_code):
    if pd.isna(ruca_code):
        return 'Unknown'
    elif ruca_code <= 3:
        return 'Urban'
    elif ruca_code <= 6:
        return 'Suburban'
    else:
        return 'Rural'

spills['ruca_code'] = pd.to_numeric(spills['ruca_code'], errors='coerce')
spills['rurality'] = spills['ruca_code'].apply(classify_rurality)
spills = spills[spills['rurality'] != 'Unknown']  # drop unknowns
